# 路径规划调用与路线展示

这个 Notebook 演示如何复用 `RoutePlanner`、比较 Dijkstra/A*/CH 的路线，并画出路径。CH 使用 C++ 动态库；运行时会通过 `05_CH/cpp/build_ch.py` 构建或更新 DLL。这里用小型内存图说明单路线调用，不读取或改写三张大型 GraphML。三规模性能对比由 `performance_service.py` 现场生成起终点并运行算法。

命令行小图演示 `06_算法调用与出图/examples/demo_route_output.py` 保留作快检；一般调用和展示以本 Notebook 为主。

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / 'workspaces' / 'A3_性能').is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError('无法定位 route_plan 仓库根目录')
A3_DIR = REPO_ROOT / 'workspaces' / 'A3_性能'
CALLS_DIR = A3_DIR / '06_算法调用与出图'
CH_CPP_DIR = A3_DIR / '05_CH' / 'cpp'
sys.path.insert(0, str(CALLS_DIR / '前端接口'))
sys.path.insert(0, str(CH_CPP_DIR))
from route_planner import RoutePlanner
from build_ch import build_ch_library

CH_LIBRARY = build_ch_library()
print('CH DLL:', CH_LIBRARY)

# 与测试共用的数据契约：连续节点 ID、CSR 数组及按 edge_id 对齐的边记录。
graph_data = {
    'n_nodes': 5,
    'indptr': [0, 2, 3, 4, 4, 4],
    'indices': [1, 2, 2, 3],
    'weights': [10.0, 50.0, 10.0, 10.0],
    'edge_ids': [0, 1, 2, 3],
    'lon': [114.0, 114.00009, 114.00018, 114.00027, 115.0],
    'lat': [30.0, 30.0, 30.0, 30.0, 31.0],
    'edges': [
        (0, 1, 10.0, 'residential', '路段甲', 36.0),
        (0, 2, 50.0, 'primary', '直达路段', 36.0),
        (1, 2, 10.0, 'residential', '路段乙', 36.0),
        (2, 3, 10.0, 'primary', '路段丙', 36.0),
    ],
}


In [ ]:
planner = RoutePlanner(graph_data, max_landmarks=2, cpp_libraries={'ch': CH_LIBRARY})
try:
    dijkstra_route = planner.plan(0, 3, algorithm='dijkstra', version='L4')
    astar_route = planner.plan(0, 3, algorithm='astar', version='L4', heuristic='H0')
    ch_route = planner.plan(0, 3, algorithm='ch', language='cpp')
    assert dijkstra_route['distance_m'] == astar_route['distance_m'] == 30.0
    print('CH      :', ch_route['node_ids'], ch_route['distance_m'], 'm')
    print('CH stats:', ch_route['stats'])
    print('Dijkstra:', dijkstra_route['node_ids'], dijkstra_route['distance_m'], 'm')
    print('A*      :', astar_route['node_ids'], astar_route['distance_m'], 'm')
    print('输出字段:', ', '.join(dijkstra_route.keys()))
finally:
    planner.close()


In [ ]:
coordinates = list(zip(graph_data['lon'], graph_data['lat']))
fig, ax = plt.subplots(figsize=(8, 3.5))
for source, target, *_ in graph_data['edges']:
    x0, y0 = coordinates[source]
    x1, y1 = coordinates[target]
    ax.plot([x0, x1], [y0, y1], color='#aab4c0', linewidth=1.5, zorder=1)
for source, target in zip(dijkstra_route['node_ids'], dijkstra_route['node_ids'][1:]):
    x0, y0 = coordinates[source]
    x1, y1 = coordinates[target]
    ax.plot([x0, x1], [y0, y1], color='#d1495b', linewidth=3, zorder=2)
for source, target in zip(ch_route['node_ids'], ch_route['node_ids'][1:]):
    x0, y0 = coordinates[source]
    x1, y1 = coordinates[target]
    ax.plot([x0, x1], [y0, y1], color='#167d8d', linewidth=4, linestyle='--', zorder=2.5, label='CH')
ax.scatter(graph_data['lon'], graph_data['lat'], color='#246a9b', zorder=3)
for node, (lon, lat) in enumerate(coordinates):
    ax.annotate(str(node), (lon, lat), xytext=(4, 5), textcoords='offset points')
ax.set_title('Dijkstra and CH routes: 0 → 3')
ax.legend()
ax.set_xlabel('longitude')
ax.set_ylabel('latitude')
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()